## Лабораторная номер 1
Выполнила: Диш Софья группа 6403

### Решите следующие задачи для данных велопарковок Сан-Франциско (trips.csv, stations.csv):
1. Найти велосипед с максимальным временем пробега.
2. Найти наибольшее геодезическое расстояние между станциями.
3. Найти путь велосипеда с максимальным временем пробега через станции.
4. Найти количество велосипедов в системе.
5. Найти пользователей потративших на поездки более 3 часов.

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("L1_Apache_Spark")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [3]:
from google.colab import files
uploaded = files.upload()

Saving trips.csv to trips.csv


In [5]:
from google.colab import files
uploaded = files.upload()

Saving stations.csv to stations.csv


In [6]:
def find_file(filename: str) -> Path:
    candidates = [
        Path.cwd() / filename,
        Path.cwd() / "data" / filename,
        Path.cwd().parent / "data" / filename,
        Path("/mnt/data") / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Не найден файл {filename}. Проверены пути: " +
        ", ".join(str(path) for path in candidates)
    )

trips_csv = find_file("trips.csv")
stations_csv = find_file("stations.csv")

trip_raw = spark.read.option("header", True).csv(str(trips_csv))
stations_raw = spark.read.option("header", True).csv(str(stations_csv))

trips_df = (
    trip_raw
    .select(
        F.col("id").cast("int").alias("id"),
        F.col("duration").cast("int").alias("duration"),
        F.coalesce(
            F.to_timestamp("start_date", "M/d/yyyy H:mm"),
            F.to_timestamp("start_date", "M/d/yyyy H:mm:ss")
        ).alias("start_date"),
        F.col("start_station_name").alias("start_station_name"),
        F.col("start_station_id").cast("int").alias("start_station_id"),
        F.coalesce(
            F.to_timestamp("end_date", "M/d/yyyy H:mm"),
            F.to_timestamp("end_date", "M/d/yyyy H:mm:ss")
        ).alias("end_date"),
        F.col("end_station_name").alias("end_station_name"),
        F.col("end_station_id").cast("int").alias("end_station_id"),
        F.col("bike_id").cast("int").alias("bike_id"),
        F.trim(F.col("subscription_type")).alias("subscription_type"),
        F.trim(F.col("zip_code")).alias("zip_code"),
    )
)

stations_df = (
    stations_raw
    .select(
        F.col("id").cast("int").alias("id"),
        F.col("name").alias("name"),
        F.col("lat").cast("double").alias("lat"),
        F.col("long").cast("double").alias("long"),
        F.col("dock_count").cast("int").alias("dock_count"),
        F.col("city").alias("city"),
        F.coalesce(
            F.to_date("installation_date", "M/d/yyyy"),
            F.to_date("installation_date", "yyyy-MM-dd")
        ).alias("installation_date"),
    )
)

In [7]:
trips_df.printSchema()
stations_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: date (nullable = true)



In [8]:
trips_df = (
    trips_df
    .where(F.col("id").isNotNull())
    .where(F.col("duration").isNotNull())
    .where(F.col("bike_id").isNotNull())
    .cache()
)

stations_df = (
    stations_df
    .where(F.col("id").isNotNull())
    .where(F.col("lat").isNotNull())
    .where(F.col("long").isNotNull())
    .cache()
)

print(f"Количество строк в trips_df: {trips_df.count()}")
print(f"Количество строк в stations_df: {stations_df.count()}")

Количество строк в trips_df: 669958
Количество строк в stations_df: 70


1. Найти велосипед с максимальным временем пробега.

In [9]:
bike_usage_summary = (
    trips_df
    .groupBy("bike_id")
    .agg(F.sum("duration").alias("total_time_sec"))
    .orderBy(F.col("total_time_sec").desc(), F.col("bike_id").asc())
)

top_bike_record = bike_usage_summary.first()

top_bike_id = top_bike_record[0]
top_bike_seconds = top_bike_record[1]
top_bike_hours = round(top_bike_seconds / 3600, 2)

print(f"Велосипед-лидер по времени использования: {top_bike_id}")
print(f"Общее время в пути: {top_bike_seconds} сек.")
print(f"То же самое в часах: {top_bike_hours} ч.")

Велосипед-лидер по времени использования: 535
Общее время в пути: 18611693 сек.
То же самое в часах: 5169.91 ч.


2. Найти наибольшее геодезическое расстояние между станциями.

In [10]:
EARTH_RADIUS_KM = 6371.0088

stations_left = stations_df.select(
    F.col("id").alias("station_id_1"),
    F.col("name").alias("station_name_1"),
    F.col("city").alias("station_city_1"),
    F.radians(F.col("lat")).alias("lat1_rad"),
    F.radians(F.col("long")).alias("lon1_rad"),
)

stations_right = stations_df.select(
    F.col("id").alias("station_id_2"),
    F.col("name").alias("station_name_2"),
    F.col("city").alias("station_city_2"),
    F.radians(F.col("lat")).alias("lat2_rad"),
    F.radians(F.col("long")).alias("lon2_rad"),
)

d_lat = F.col("lat2_rad") - F.col("lat1_rad")
d_lon = F.col("lon2_rad") - F.col("lon1_rad")

haversine_a = (
    F.pow(F.sin(d_lat / 2), 2) +
    F.cos(F.col("lat1_rad")) * F.cos(F.col("lat2_rad")) * F.pow(F.sin(d_lon / 2), 2)
)

distance_km_expr = F.lit(2 * EARTH_RADIUS_KM) * F.asin(F.sqrt(haversine_a))

farthest_pair = (
    stations_left.crossJoin(stations_right)
    .filter(F.col("station_id_1") < F.col("station_id_2"))
    .withColumn("geo_distance_km", distance_km_expr)
    .orderBy(F.col("geo_distance_km").desc())
    .limit(1)
    .collect()[0]
)

print("Самая удалённая пара станций:")
print(f"Станция 1: {farthest_pair['station_name_1']} ({farthest_pair['station_city_1']})")
print(f"Станция 2: {farthest_pair['station_name_2']} ({farthest_pair['station_city_2']})")
print(f"Расстояние между ними: {farthest_pair['geo_distance_km']:.2f} км")

Самая удалённая пара станций:
Станция 1: SJSU - San Salvador at 9th (San Jose)
Станция 2: Embarcadero at Sansome (San Francisco)
Расстояние между ними: 69.92 км


3. Найти путь велосипеда с максимальным временем пробега через станции.

In [12]:
leader_trips = (
    trips_df
    .filter(F.col("bike_id") == top_bike_id)
    .select(
        "id",
        "start_date",
        "start_station_name",
        "end_date",
        "end_station_name",
        "duration"
    )
    .orderBy("start_date", "end_date", "id")
)

total_rides = leader_trips.count()

print(f"История перемещений велосипеда #{top_bike_id} ({total_rides} поездок):")
leader_trips.show(total_rides, truncate=False)

trip_segments = leader_trips.select("start_station_name", "end_station_name").collect()

if trip_segments:
    full_route_stations = [trip_segments[0]["start_station_name"]]
    full_route_stations.extend([row["end_station_name"] for row in trip_segments])

    print("\nПолная последовательность посещенных станций:")
    print(" ".join(full_route_stations))
else:
    print(f"Для велосипеда #{top_bike_id} не найдено ни одной поездки.")

История перемещений велосипеда #535 (1328 поездок):
+------+-------------------+---------------------------------------------+-------------------+---------------------------------------------+--------+
|id    |start_date         |start_station_name                           |end_date           |end_station_name                             |duration|
+------+-------------------+---------------------------------------------+-------------------+---------------------------------------------+--------+
|4966  |2013-08-29 19:32:00|Post at Kearney                              |2013-08-29 19:53:00|San Francisco Caltrain (Townsend at 4th)     |1245    |
|5067  |2013-08-29 21:38:00|San Francisco Caltrain (Townsend at 4th)     |2013-08-29 21:45:00|San Francisco Caltrain 2 (330 Townsend)      |423     |
|5179  |2013-08-30 08:40:00|San Francisco Caltrain 2 (330 Townsend)      |2013-08-30 08:54:00|Market at Sansome                            |842     |
|5199  |2013-08-30 09:10:00|Market at Sansome   

4. Найти количество велосипедов в системе.

In [13]:
unique_bikes_count = trips_df.agg(F.countDistinct("bike_id").alias("total_bikes")).collect()[0]["total_bikes"]

print(f"Всего зарегистрировано велосипедов: {unique_bikes_count}")

Всего зарегистрировано велосипедов: 700


5. Найти пользователей потративших на поездки более 3 часов.

In [14]:
TIME_THRESHOLD_SEC = 3 * 60 * 60

valid_trips = trips_df.filter(
    F.col("zip_code").isNotNull() & (F.col("zip_code") != "")
)

zip_stats = (
    valid_trips
    .groupBy("zip_code")
    .agg(F.sum("duration").alias("total_time_sec"))
    .filter(F.col("total_time_sec") > TIME_THRESHOLD_SEC)
    .withColumn("total_time_hours", F.round(F.col("total_time_sec") / 3600, 2))
    .orderBy(F.desc("total_time_sec"), F.asc("zip_code"))
)

print("Индексы (zip_code), владельцы которых провели в седле более 3 часов:")
zip_stats.show(truncate=False)

active_zip_codes = [row["zip_code"] for row in zip_stats.select("zip_code").collect()]

if active_zip_codes:
    result_string = ", ".join(active_zip_codes)
    print(f"Найденные активные зоны: {result_string}")
else:
    print("В данном наборе данных нет пользователей с общим временем поездок > 3 часов.")

spark.stop()

Индексы (zip_code), владельцы которых провели в седле более 3 часов:
+--------+--------------+----------------+
|zip_code|total_time_sec|total_time_hours|
+--------+--------------+----------------+
|94107   |49757162      |13821.43        |
|nil     |45725550      |12701.54        |
|94105   |25596128      |7110.04         |
|94133   |21637675      |6010.47         |
|94102   |19128021      |5313.34         |
|94103   |19127388      |5313.16         |
|95531   |17270400      |4797.33         |
|94111   |14244997      |3956.94         |
|95112   |12742370      |3539.55         |
|94109   |12057128      |3349.2          |
|94040   |7807926       |2168.87         |
|94110   |7421936       |2061.65         |
|94117   |6901313       |1917.03         |
|94301   |6590378       |1830.66         |
|94041   |6276284       |1743.41         |
|94158   |6248167       |1735.6          |
|94306   |5550643       |1541.85         |
|94025   |5178237       |1438.4          |
|94108   |5127562       |142